## Parte 1 — Softmax con temperatura

Se implementó en NumPy la softmax con temperatura de forma numéricamente estable: antes de exponenciar se resta el máximo de los logits, es decir, p_i = exp((z_i − m)/T) / Σ_j exp((z_j − m)/T) con m = max_j z_j, lo que evita desbordamientos sin alterar el resultado. La entropía se calculó en bits como H = −Σ_i p_i log2(p_i). Con los logits z = [2.0, 1.0, 0.5, 0.2, −1.0, −3.0] para t0…t5, las tres distribuciones (cada una suma 1, verificado en la celda) y sus entropías son:

| Token | T = 0.5 | T = 1.0 | T = 2.0 |
|---|---|---|---|
| t0 | 0.8231 | 0.5516 | 0.3583 |
| t1 | 0.1114 | 0.2029 | 0.2173 |
| t2 | 0.0410 | 0.1231 | 0.1693 |
| t3 | 0.0225 | 0.0912 | 0.1457 |
| t4 | 0.0020 | 0.0275 | 0.0800 |
| t5 | 0.0000 | 0.0037 | 0.0294 |
| **H (bits)** | **0.9147** | **1.7998** | **2.2888** |

La temperatura actúa como control de concentración: al bajar de T = 2 a T = 0.5 la masa se concentra en t0 (0.3583 → 0.8231) y la entropía cae de 2.2888 a 0.9147 bits; al subirla, la distribución se aplana y la entropía crece monótonamente (0.9147 → 1.7998 → 2.2888 bits). Nótese que el 0.0000 de t5 con T = 0.5 corresponde a 3.7367e-05 antes de redondear.

In [1]:
# T1: código aprobado por el crítico
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------
# Datos del enunciado (Parte 1 — Softmax con temperatura)
# ----------------------------------------------------------------------
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
temperaturas = [0.5, 1.0, 2.0]


def softmax_con_temperatura(z, T):
    """Softmax con temperatura p_i = exp(z_i/T) / sum_j exp(z_j/T).

    Numéricamente estable: se resta el máximo de z/T antes de exponenciar,
    de modo que el mayor exponente es 0 y no hay desbordamiento.
    """
    z = np.asarray(z, dtype=float)
    escalados = z / float(T)
    escalados = escalados - np.max(escalados)  # estabilidad numérica
    e = np.exp(escalados)
    return e / np.sum(e)


def entropia_bits(p):
    """Entropía de Shannon en bits: H = -sum p_i log2(p_i), con 0·log2(0)=0."""
    p = np.asarray(p, dtype=float)
    p_pos = p[p > 0]
    return float(-np.sum(p_pos * np.log2(p_pos)))


# ----------------------------------------------------------------------
# Cálculo de distribuciones y entropías
# ----------------------------------------------------------------------
distribuciones = {T: softmax_con_temperatura(logits, T) for T in temperaturas}
entropias = {T: entropia_bits(distribuciones[T]) for T in temperaturas}
sumas = {T: float(np.sum(distribuciones[T])) for T in temperaturas}

# Verificaciones del criterio de éxito
sumas_ok = all(abs(sumas[T] - 1.0) < 1e-12 for T in temperaturas)
monotona = bool(entropias[0.5] < entropias[1.0] < entropias[2.0])

# ----------------------------------------------------------------------
# Tabla con cuatro decimales (distribuciones + entropía en bits)
# ----------------------------------------------------------------------
cols = [f"T={T}" for T in temperaturas]
tabla = pd.DataFrame(
    {c: [f"{distribuciones[T][i]:.4f}" for i in range(len(tokens))]
     for c, T in zip(cols, temperaturas)},
    index=tokens,
)
tabla.loc["H (bits)"] = [f"{entropias[T]:.4f}" for T in temperaturas]

# ----------------------------------------------------------------------
# Figura: distribuciones para cada temperatura
# ----------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(tokens))
ancho = 0.25
for i, T in enumerate(temperaturas):
    ax.bar(x + (i - 1) * ancho, distribuciones[T], width=ancho, label=f"T = {T}")
ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_ylabel("Probabilidad")
ax.set_xlabel("Token")
ax.set_title("Softmax con temperatura (logits: 2.0, 1.0, 0.5, 0.2, -1.0, -3.0)")
ax.legend()
fig.tight_layout()
plt.savefig("softmax_temperatura.png", dpi=120)
plt.close(fig)

# ----------------------------------------------------------------------
# Contrato: resultados.json (valores sin redondear + tabla a 4 decimales)
# ----------------------------------------------------------------------
resultados = {
    "tokens": tokens,
    "logits": logits.tolist(),
    "temperaturas": temperaturas,
    "distribuciones": {f"T={T}": distribuciones[T].tolist() for T in temperaturas},
    "entropias_bits": {f"T={T}": entropias[T] for T in temperaturas},
    "suma_de_cada_distribucion": {f"T={T}": sumas[T] for T in temperaturas},
    "verificacion_suma_1": bool(sumas_ok),
    "entropia_monotona_creciente_con_T": monotona,
    "tabla_4_decimales": {
        "columnas": ["token"] + cols,
        "filas": (
            [[tok] + [f"{distribuciones[T][i]:.4f}" for T in temperaturas]
             for i, tok in enumerate(tokens)]
            + [["H (bits)"] + [f"{entropias[T]:.4f}" for T in temperaturas]]
        ),
    },
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# Tabla también en CSV (opcional, solo la presentación a 4 decimales)
tabla.to_csv("tabla_softmax_temperatura.csv")

# ----------------------------------------------------------------------
# Resumen
# ----------------------------------------------------------------------
print("Softmax con temperatura — Tarea B, Parte 1")
print(f"Logits: {logits.tolist()}")
print()
print(tabla.to_string())
print()
for T in temperaturas:
    print(f"T={T}: suma={sumas[T]:.12f}  H={entropias[T]:.10f} bits")
print(f"Sumas iguales a 1: {sumas_ok}")
print(f"Entropía monótona creciente con T (H(0.5) < H(1) < H(2)): {monotona}")
print("Archivos generados: resultados.json, softmax_temperatura.png, tabla_softmax_temperatura.csv")


Softmax con temperatura — Tarea B, Parte 1
Logits: [2.0, 1.0, 0.5, 0.2, -1.0, -3.0]

           T=0.5   T=1.0   T=2.0
t0        0.8231  0.5516  0.3583
t1        0.1114  0.2029  0.2173
t2        0.0410  0.1231  0.1693
t3        0.0225  0.0912  0.1457
t4        0.0020  0.0275  0.0800
t5        0.0000  0.0037  0.0294
H (bits)  0.9147  1.7998  2.2888

T=0.5: suma=1.000000000000  H=0.9146817686 bits
T=1.0: suma=1.000000000000  H=1.7998355411 bits
T=2.0: suma=1.000000000000  H=2.2888261658 bits
Sumas iguales a 1: True
Entropía monótona creciente con T (H(0.5) < H(1) < H(2)): True
Archivos generados: resultados.json, softmax_temperatura.png, tabla_softmax_temperatura.csv


## Parte 2 — Muestreo de núcleo (top-p)

Sobre la distribución de T = 1 de la Parte 1 se ordenaron los tokens de mayor a menor probabilidad y se acumuló: t0 (0.5516), t1 (acumulada 0.7546), t2 (0.8776), t3 (0.9688), t4 (0.9963) y t5 (1.0000). El conjunto mínimo cuya acumulada alcanza p = 0.9 es {t0, t1, t2, t3}: con solo t0, t1 y t2 la acumulada es 0.8776 < 0.9, y al añadir t3 se llega a 0.9688 ≥ 0.9; por eso el núcleo acumula 0.9688, algo por encima del umbral. Los tokens t4 y t5 quedan fuera del núcleo con probabilidad 0, y las probabilidades de los supervivientes se renormalizan dividiendo entre 0.9688:

| Token | p (T = 1) | Acumulada | ¿En el núcleo? | p renormalizada |
|---|---|---|---|---|
| t0 | 0.5516 | 0.5516 | sí | 0.5694 |
| t1 | 0.2029 | 0.7546 | sí | 0.2095 |
| t2 | 0.1231 | 0.8776 | sí | 0.1270 |
| t3 | 0.0912 | 0.9688 | sí | 0.0941 |
| t4 | 0.0275 | 0.9963 | no | 0.0000 |
| t5 | 0.0037 | 1.0000 | no | 0.0000 |

**Tokens supervivientes: t0, t1, t2, t3.** La renormalización eleva ligeramente cada probabilidad (t0 pasa de 0.5516 a 0.5694) porque la masa eliminada de t4 y t5 se reparte proporcionalmente entre el núcleo; la distribución resultante suma 1 (verificado en la celda).

In [2]:
# T2: código aprobado por el crítico
# -*- coding: utf-8 -*-
"""
T2 — Muestreo de núcleo (top-p) con p = 0.9 sobre la distribución de T = 1.

Entrada : entrada/T1/resultados.json  (distribución softmax con T=1 de la Parte 1)
Salidas : resultados.json             (contrato de la subtarea)
          T2_top_p_renormalizada.png  (figura comparativa)
"""

import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

P_UMBRAL = 0.9
TOKENS = ["t0", "t1", "t2", "t3", "t4", "t5"]
LOGITS = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0])


# ----------------------------------------------------------------------
# 1) Distribución de T = 1 de la Parte 1 (se lee de entrada/T1; si no
#    estuviera disponible, se recalcula con softmax estable).
# ----------------------------------------------------------------------
def softmax_con_temperatura(logits, T):
    z = np.asarray(logits, dtype=float) / float(T)
    z = z - z.max()  # estabilidad numérica
    e = np.exp(z)
    return e / e.sum()


ruta_t1 = Path("entrada") / "T1" / "resultados.json"
fuente = "recalculada con softmax(T=1) a partir de los logits del enunciado"
if ruta_t1.exists():
    with open(ruta_t1, "r", encoding="utf-8") as f:
        res_t1 = json.load(f)
    p_t1 = np.array(res_t1["distribuciones"]["T=1.0"], dtype=float)
    fuente = "entrada/T1/resultados.json -> distribuciones['T=1.0']"
    # Verificación: debe coincidir con el softmax numéricamente estable
    assert np.allclose(p_t1, softmax_con_temperatura(LOGITS, 1.0), atol=1e-12)
else:
    p_t1 = softmax_con_temperatura(LOGITS, 1.0)

# ----------------------------------------------------------------------
# 2) Top-p: ordenar descendente, conjunto mínimo con acumulada >= p,
#    renormalizar las probabilidades del núcleo.
# ----------------------------------------------------------------------
orden = np.argsort(-p_t1, kind="stable")            # índices de mayor a menor
tokens_ord = [TOKENS[i] for i in orden]
probs_ord = p_t1[orden]
acum_ord = np.cumsum(probs_ord)

# k = número mínimo de tokens cuya probabilidad acumulada alcanza p = 0.9
k = int(np.searchsorted(acum_ord, P_UMBRAL, side="left")) + 1
idx_nucleo = orden[:k]
nucleo_set = set(idx_nucleo.tolist())
tokens_nucleo = [TOKENS[i] for i in idx_nucleo]
probs_nucleo = p_t1[idx_nucleo]
acum_nucleo = float(acum_ord[k - 1])

# Verificación de minimalidad: con k-1 tokens NO se alcanza p
assert acum_nucleo >= P_UMBRAL
assert (k == 1) or (float(acum_ord[k - 2]) < P_UMBRAL)

# Renormalización
probs_renorm_nucleo = probs_nucleo / probs_nucleo.sum()
p_renorm = np.zeros_like(p_t1)
p_renorm[idx_nucleo] = probs_renorm_nucleo

# ----------------------------------------------------------------------
# 3) Verificaciones del criterio de éxito
# ----------------------------------------------------------------------
suma_renorm = float(p_renorm.sum())
tokens_fuera = [t for i, t in enumerate(TOKENS) if i not in nucleo_set]
checks = {
    "suma_de_la_renormalizada_es_1": bool(abs(suma_renorm - 1.0) < 1e-12),
    "acumulada_del_nucleo_alcanza_p": bool(acum_nucleo >= P_UMBRAL),
    "es_el_conjunto_minimo": bool((k == 1) or (float(acum_ord[k - 2]) < P_UMBRAL)),
    "tokens_fuera_del_nucleo_con_prob_0": bool(
        all(p_renorm[i] == 0.0 for i in range(len(TOKENS)) if i not in nucleo_set)
    ),
    "todas_las_probs_son_no_negativas": bool(np.all(p_renorm >= 0.0)),
}
assert all(checks.values()), checks

# ----------------------------------------------------------------------
# 4) Tabla ordenada y resultados.json (contrato)
# ----------------------------------------------------------------------
tabla = []
for pos, i in enumerate(orden):
    tabla.append({
        "token": TOKENS[i],
        "probabilidad_T1": float(p_t1[i]),
        "acumulada": float(acum_ord[pos]),
        "en_nucleo": bool(i in nucleo_set),
        "probabilidad_renormalizada": float(p_renorm[i]),
    })

resultados = {
    "subtarea": "T2 — Muestreo de núcleo (top-p) con p = 0.9 sobre la distribución de T=1",
    "p": P_UMBRAL,
    "fuente_distribucion_entrada": fuente,
    "distribucion_entrada_T1": {t: float(v) for t, v in zip(TOKENS, p_t1)},
    "tokens_ordenados_descendente": tokens_ord,
    "probabilidades_ordenadas": [float(v) for v in probs_ord],
    "acumuladas_ordenadas": [float(v) for v in acum_ord],
    "tokens_supervivientes": tokens_nucleo,
    "num_tokens_en_nucleo": k,
    "probabilidad_acumulada_nucleo": acum_nucleo,
    "probabilidades_nucleo_antes_de_renormalizar": {
        t: float(v) for t, v in zip(tokens_nucleo, probs_nucleo)
    },
    "distribucion_renormalizada_nucleo": {
        t: float(v) for t, v in zip(tokens_nucleo, probs_renorm_nucleo)
    },
    "distribucion_renormalizada_vocabulario_completo": {
        t: float(v) for t, v in zip(TOKENS, p_renorm)
    },
    "tokens_fuera_del_nucleo": tokens_fuera,
    "suma_distribucion_renormalizada": suma_renorm,
    "verificaciones": checks,
    "tabla_ordenada": tabla,
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

# ----------------------------------------------------------------------
# 5) Figura: distribución T=1 original vs top-p renormalizada
# ----------------------------------------------------------------------
x = np.arange(len(TOKENS))
w = 0.38
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - w / 2, p_t1, width=w, color="#4C72B0", label="Softmax T=1 (original)")
ax.bar(x + w / 2, p_renorm, width=w, color="#DD8452", label="Top-p p=0.9 (renormalizada)")
for i in range(len(TOKENS)):
    ax.text(x[i] - w / 2, p_t1[i] + 0.008, f"{p_t1[i]:.4f}", ha="center", fontsize=8)
    if p_renorm[i] > 0:
        ax.text(x[i] + w / 2, p_renorm[i] + 0.008, f"{p_renorm[i]:.4f}", ha="center", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(TOKENS)
ax.set_ylabel("Probabilidad")
ax.set_ylim(0, 1.05 * max(p_t1.max(), p_renorm.max()) + 0.05)
ax.set_title("Top-p (nucleus sampling), p = 0.9 — núcleo: " + ", ".join(tokens_nucleo))
ax.legend()
fig.tight_layout()
fig.savefig("T2_top_p_renormalizada.png", dpi=120)
plt.close(fig)

# ----------------------------------------------------------------------
# 6) Resumen
# ----------------------------------------------------------------------
print("=" * 64)
print("T2 — Top-p con p = 0.9 sobre la distribución de T = 1")
print("=" * 64)
print("Distribución de entrada (T=1):")
for t, v in zip(TOKENS, p_t1):
    print(f"  {t}: {v:.6f}")
print(f"\nOrden descendente: {tokens_ord}")
print("Acumuladas: " + ", ".join(f"{a:.4f}" for a in acum_ord))
print(f"\nNúcleo (mínimo conjunto con acumulada >= 0.9): {tokens_nucleo}")
print(f"  Tokens conservados: {k} | Acumulada del núcleo: {acum_nucleo:.6f}")
print(f"  Tokens fuera del núcleo (prob. 0): {tokens_fuera}")
print("\nDistribución renormalizada (suma = {:.12f}):".format(suma_renorm))
for t, v in zip(TOKENS, p_renorm):
    print(f"  {t}: {v:.6f}")
print("\nArchivos escritos: resultados.json, T2_top_p_renormalizada.png")


T2 — Top-p con p = 0.9 sobre la distribución de T = 1
Distribución de entrada (T=1):
  t0: 0.551623
  t1: 0.202931
  t2: 0.123084
  t3: 0.091183
  t4: 0.027464
  t5: 0.003717

Orden descendente: ['t0', 't1', 't2', 't3', 't4', 't5']
Acumuladas: 0.5516, 0.7546, 0.8776, 0.9688, 0.9963, 1.0000

Núcleo (mínimo conjunto con acumulada >= 0.9): ['t0', 't1', 't2', 't3']
  Tokens conservados: 4 | Acumulada del núcleo: 0.968820
  Tokens fuera del núcleo (prob. 0): ['t4', 't5']

Distribución renormalizada (suma = 1.000000000000):
  t0: 0.569376
  t1: 0.209462
  t2: 0.127045
  t3: 0.094117
  t4: 0.000000
  t5: 0.000000

Archivos escritos: resultados.json, T2_top_p_renormalizada.png


## Parte 3 — Comprobación empírica

Se tomaron 10,000 muestras de la distribución renormalizada de la Parte 2 con numpy.random.default_rng(0). Las frecuencias observadas frente a las probabilidades teóricas son:

| Token | Teórica | Observada | Conteo (de 10,000) |
|---|---|---|---|
| t0 | 0.5694 | 0.5682 | 5682 |
| t1 | 0.2095 | 0.2093 | 2093 |
| t2 | 0.1270 | 0.1296 | 1296 |
| t3 | 0.0941 | 0.0929 | 929 |
| t4 | 0.0000 | 0.0000 | 0 |
| t5 | 0.0000 | 0.0000 | 0 |

La divergencia KL de la distribución empírica respecto de la teórica es **5.0069e-05 bits**, un valor prácticamente nulo: con 10,000 muestras las frecuencias reproducen con fidelidad la distribución objetivo. t4 y t5 nunca aparecieron (0 de 10,000), coherente con su probabilidad 0 tras el top-p. Por token, la mayor contribución positiva a la KL es la de t2 (0.0037 bits), sobrerrepresentado (0.1296 observado frente a 0.1270 teórico), compensada en parte por las contribuciones negativas de t0 (−0.0017), t3 (−0.0017) y t1 (−0.0002); la suma total se mantiene no negativa. La celda de esta parte genera el gráfico de barras que compara las frecuencias observadas con las teóricas.

In [3]:
# T3: código aprobado por el crítico
# -*- coding: utf-8 -*-
"""
T3 — Comprobación empírica del muestreo de núcleo (top-p, p = 0.9).

Toma 10,000 muestras de la distribución renormalizada de la Parte 2 (resultado de T2)
usando numpy.random.default_rng(0), compara las frecuencias observadas con las
probabilidades teóricas en un gráfico de barras y calcula la divergencia KL
(en bits) de la distribución empírica respecto de la teórica:

    KL(emp || teo) = sum_i emp_i * log2(emp_i / teo_i)

Salidas:
  - resultados.json (contrato de la subtarea)
  - T3_frecuencias_observadas_vs_teoricas.png
"""

import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

TOKENS = [f"t{i}" for i in range(6)]
N_MUESTRAS = 10_000
SEMILLA = 0
P_TOP = 0.9

# ------------------------------------------------------------------ entrada
# Fuente principal: resultado de T2. Respaldo: recalcular desde los logits del enunciado.
ruta_t2 = Path("entrada/T2/resultados.json")
if ruta_t2.exists():
    with open(ruta_t2, "r", encoding="utf-8") as f:
        res_t2 = json.load(f)
    dist_teo_dict = res_t2["distribucion_renormalizada_vocabulario_completo"]
    p_teo = np.array([dist_teo_dict[t] for t in TOKENS], dtype=float)
    nucleo = list(res_t2["tokens_supervivientes"])
    fuente = "entrada/T2/resultados.json -> distribucion_renormalizada_vocabulario_completo"
else:
    logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
    z = logits - logits.max()                      # softmax T=1 numéricamente estable
    p1 = np.exp(z) / np.exp(z).sum()
    orden = np.argsort(-p1)                        # descendente
    acum = np.cumsum(p1[orden])
    k = int(np.searchsorted(acum, P_TOP, side="left") + 1)  # conjunto mínimo que alcanza p
    idx_nuc = np.sort(orden[:k])
    p_nuc = p1[idx_nuc] / p1[idx_nuc].sum()        # renormalización
    p_teo = np.zeros(len(TOKENS), dtype=float)
    p_teo[idx_nuc] = p_nuc
    nucleo = [TOKENS[i] for i in idx_nuc]
    fuente = "recalculada desde logits (softmax T=1 + top-p p=0.9)"

en_nucleo = np.array([t in nucleo for t in TOKENS])

# ------------------------------------------------------------------ muestreo
rng = np.random.default_rng(SEMILLA)
# Normalización numérica (~1e-16) solo para que rng.choice acepte p; no altera la distribución.
p_muestreo = p_teo / p_teo.sum()
muestras = rng.choice(len(TOKENS), size=N_MUESTRAS, p=p_muestreo)

conteos = np.bincount(muestras, minlength=len(TOKENS))
f_obs = conteos / N_MUESTRAS

# ------------------------------------------------------------------ divergencia KL (bits)
# KL(emp || teo) = sum_i emp_i * log2(emp_i / teo_i).
# Convención: términos con emp_i = 0 aportan 0 (aquí solo t4, t5, con teo_i = 0 también).
mask = f_obs > 0
kl_bits = float(np.sum(f_obs[mask] * np.log2(f_obs[mask] / p_teo[mask])))

contrib_kl = {
    t: (float(fq * np.log2(fq / p)) if fq > 0 else 0.0)
    for t, fq, p in zip(TOKENS, f_obs, p_teo)
}

# ------------------------------------------------------------------ figura
fig, ax = plt.subplots(figsize=(9.5, 5.5))
x = np.arange(len(TOKENS))
w = 0.38
b1 = ax.bar(x - w / 2, f_obs, width=w, color="#4C72B0", edgecolor="black", linewidth=0.5,
            label=f"Frecuencia observada ({N_MUESTRAS:,} muestras, semilla {SEMILLA})")
b2 = ax.bar(x + w / 2, p_teo, width=w, color="#DD8452", edgecolor="black", linewidth=0.5,
            label="Probabilidad teórica (top-p renormalizada)")
for barras in (b1, b2):
    for rect in barras:
        h = rect.get_height()
        ax.annotate(f"{h:.4f}",
                    (rect.get_x() + rect.get_width() / 2, h),
                    xytext=(0, 2), textcoords="offset points",
                    ha="center", va="bottom", fontsize=8)

ax.axvline(len(nucleo) - 0.5, color="gray", linestyle="--", linewidth=1)
ax.text(len(nucleo) - 0.5 + 0.1, max(f_obs.max(), p_teo.max()) * 0.97,
        "fuera del núcleo\n(p = 0)", fontsize=8, color="gray", va="top")

ax.set_xticks(x)
ax.set_xticklabels(TOKENS)
ax.set_xlabel("Token")
ax.set_ylabel("Probabilidad / frecuencia")
ax.set_ylim(0, max(f_obs.max(), p_teo.max()) * 1.18)
ax.set_title(
    "Parte 3 — Comprobación empírica del muestreo de núcleo (top-p, p = 0.9)\n"
    f"Frecuencias observadas vs. probabilidades teóricas | n = {N_MUESTRAS:,}, "
    f"semilla {SEMILLA} | KL(emp||teo) = {kl_bits:.6f} bits"
)
ax.legend(loc="upper right", fontsize=9)
fig.tight_layout()
fig.savefig("T3_frecuencias_observadas_vs_teoricas.png", dpi=120)
plt.close(fig)

# ------------------------------------------------------------------ resultados.json
resultados = {
    "subtarea": ("T3 — Comprobación empírica: 10,000 muestras de la distribución renormalizada "
                 "de la Parte 2 (top-p, p=0.9) con numpy.random.default_rng(0); comparación "
                 "frecuencias observadas vs. probabilidades teóricas y divergencia KL en bits"),
    "n_muestras": N_MUESTRAS,
    "semilla": SEMILLA,
    "generador": "numpy.random.default_rng(0)",
    "p_top": P_TOP,
    "fuente_distribucion_entrada": fuente,
    "tokens": TOKENS,
    "tokens_en_nucleo": nucleo,
    "tokens_fuera_del_nucleo": [t for t in TOKENS if t not in nucleo],
    "distribucion_teorica": {t: float(p) for t, p in zip(TOKENS, p_teo)},
    "conteos_observados": {t: int(c) for t, c in zip(TOKENS, conteos)},
    "frecuencias_observadas": {t: float(fq) for t, fq in zip(TOKENS, f_obs)},
    "divergencia_KL_bits": kl_bits,
    "contribuciones_KL_bits_por_token": contrib_kl,
    "figura": "T3_frecuencias_observadas_vs_teoricas.png",
    "verificaciones": {
        "suma_distribucion_teorica": float(p_teo.sum()),
        "suma_frecuencias_observadas": float(f_obs.sum()),
        "total_conteos": int(conteos.sum()),
        "tokens_fuera_del_nucleo_nunca_muestreados": bool(conteos[~en_nucleo].sum() == 0),
        "kl_no_negativa": bool(kl_bits >= 0.0),
    },
}
with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ------------------------------------------------------------------ resumen
print("T3 — Comprobación empírica (top-p p=0.9, n=10,000, semilla 0)")
print(f"Distribución teórica obtenida de: {fuente}")
print(f"{'token':<6}{'p_teorica':>12}{'conteo':>9}{'f_observada':>13}")
for t, p, c, fq in zip(TOKENS, p_teo, conteos, f_obs):
    print(f"{t:<6}{p:>12.6f}{c:>9d}{fq:>13.5f}")
print(f"\nDivergencia KL(empírica || teórica) = {kl_bits:.8f} bits  (valor pequeño, cercano a 0)")
print("Figura guardada: T3_frecuencias_observadas_vs_teoricas.png")
print("Resultados escritos: resultados.json")


T3 — Comprobación empírica (top-p p=0.9, n=10,000, semilla 0)
Distribución teórica obtenida de: entrada/T2/resultados.json -> distribucion_renormalizada_vocabulario_completo
token    p_teorica   conteo  f_observada
t0        0.569376     5682      0.56820
t1        0.209462     2093      0.20930
t2        0.127045     1296      0.12960
t3        0.094117      929      0.09290
t4        0.000000        0      0.00000
t5        0.000000        0      0.00000

Divergencia KL(empírica || teórica) = 0.00005007 bits  (valor pequeño, cercano a 0)
Figura guardada: T3_frecuencias_observadas_vs_teoricas.png
Resultados escritos: resultados.json


## Parte 4 — Pregunta conceptual

**Cuando T → 0**, el cociente z_i/T amplifica las diferencias entre logits y el término exponencial del logit máximo domina exponencialmente a los demás: toda la masa se concentra en el token de mayor logit (aquí t0) y la distribución degenera en una masa de punto. Esto es exactamente la decodificación voraz (greedy), que en cada paso elige el argmax de los logits: muestrear una distribución que pone probabilidad 1 en el argmax devuelve ese mismo token con certeza, de modo que ambos procedimientos generan secuencias idénticas. Las entropías de la Parte 1 evidencian la tendencia: al reducir T la entropía cae monótonamente, 2.2888 bits (T = 2) → 1.7998 (T = 1) → 0.9147 (T = 0.5), mientras p(t0) sube 0.3583 → 0.5516 → 0.8231 y p(t5) cae 0.0294 → 0.0037 → 0.0000 (3.7367e-05 sin redondear). Extrapolando, en el límite la entropía tiende a 0 bits y p(t0) a 1: el caso plenamente determinista, equivalente a greedy.

**Cuando T → ∞**, en cambio, z_i/T → 0 para todo token, todos los exponenciales se igualan y la distribución tiende a la uniforme sobre los seis tokens: ninguno es preferido y la entropía tiende a su máximo posible para el vocabulario. Los datos apuntan en esa dirección: al pasar de T = 0.5 a T = 2 la entropía sube de 0.9147 a 2.2888 bits y las probabilidades se aplanan (t0 baja de 0.8231 a 0.3583; t5 sube de 0.0000 a 0.0294). En síntesis, T gobierna el compromiso entre determinismo y diversidad: T baja concentra la masa y reduce la entropía (hasta equivaler a la decodificación voraz), mientras que T alta la reparte uniformemente y maximiza la aleatoriedad.